# Gisborne screening: deriving the committed findings

This notebook recomputes each headline number in the project README directly
from the committed real-data outputs, so the claims can be checked without
rerunning the whole pipeline. It covers the three findings in README order:
overlay materiality, width-method disagreement, and the outstanding imagery
review. Neither width proxy is MPI's formal centre-line measurement.

In [1]:
from pathlib import Path
import json
import warnings

import geopandas as gpd
import pandas as pd

ROOT = Path('..')
OUT = ROOT / 'outputs' / 'gisborne'

candidates = gpd.read_file(ROOT / 'data/processed/gisborne_candidates.gpkg')
screened = pd.concat(
    [gpd.read_file(OUT / 'candidates.gpkg'), gpd.read_file(OUT / 'quarantine.gpkg')],
    ignore_index=True,
)
screened = gpd.GeoDataFrame(screened, geometry='geometry', crs=2193)
comparison = pd.read_csv(OUT / 'width_method_comparison.csv')
findings = json.loads((OUT / 'findings.json').read_text())

print(f'Input LCDB units:      {len(candidates):,}')
print(f'Screened dispositions: {len(screened):,}')
print(screened.status.value_counts().to_string())

Input LCDB units:      5,712
Screened dispositions: 5,712
status
quarantine          2801
candidate_review    2687
excluded             224


## Finding 1 — boundary mismatch can dominate whole-unit decisions

The earlier rule flagged any conservation intersection larger than 1 m². The
cell below separates those flags into *material* overlaps (at least 1% of the
LCDB unit) and slivers, and contrasts the **source-unit area** the slivers put
at risk against the **actual intersection area** that justified the flag.

In [2]:
def overlay_sensitivity(column_m2, column_pct, label):
    flagged = screened[column_m2] > 1.0
    material = flagged & (screened[column_pct] >= 1.0)
    sliver = flagged & ~material
    print(f'{label}:')
    print(f'  area-only flags (>1 m2):        {int(flagged.sum()):,}')
    print(f'  material at >=1% of the unit:   {int(material.sum()):,}')
    print(f'  reclassified to advisory:       {int(sliver.sum()):,}')
    print(f'  source-unit area, all flags:    {screened.loc[flagged].area.sum()/1e4:,.1f} ha')
    print(f'  source-unit area, slivers only: {screened.loc[sliver].area.sum()/1e4:,.1f} ha')
    print(f'  actual intersection, slivers:   {screened.loc[sliver, column_m2].sum()/1e4:,.1f} ha')


overlay_sensitivity('conservation_overlap_m2', 'conservation_overlap_pct', 'R-04 conservation')
print()
overlay_sensitivity('pre1990_overlap_m2', 'pre1990_overlap_pct', 'R-03 mapped pre-1990')

R-04 conservation:
  area-only flags (>1 m2):        302
  material at >=1% of the unit:   224
  reclassified to advisory:       78
  source-unit area, all flags:    283,698.7 ha
  source-unit area, slivers only: 275,035.5 ha
  actual intersection, slivers:   679.3 ha

R-03 mapped pre-1990:
  area-only flags (>1 m2):        900
  material at >=1% of the unit:   691
  reclassified to advisory:       209
  source-unit area, all flags:    318,996.9 ha
  source-unit area, slivers only: 273,861.1 ha
  actual intersection, slivers:   1,032.6 ha


78 sliver-overlap units carried 275,035.5 ha of source-unit area while their
real DOC intersection totalled 679.3 ha — a ratio of roughly 400:1. An
`intersects` test would have discarded all of that area on the strength of the
679.3 ha. This is a mapping-generalisation artefact, not a land-status finding.

The surviving advisory cases are exported per unit so an assessor can work down
them in overlap order rather than see only a count.

In [3]:
advisory = pd.read_csv(OUT / 'advisory_candidates.csv')
print(f'candidates carrying an advisory flag: {len(advisory)}')
print(advisory.advisory_rule_ids.value_counts().to_string())
print()
print(advisory.head(5).to_string(index=False))

candidates carrying an advisory flag: 167
advisory_rule_ids
R-03-low-overlap                     120
R-03-low-overlap|R-04-low-overlap     27
R-04-low-overlap                      20

       unit_id                      lcdb_class  area_ha advisory_rule_ids  pre1990_overlap_m2  pre1990_overlap_pct  conservation_overlap_m2  conservation_overlap_pct  max_overlap_pct
lcdb1000226494         Low Producing Grassland   4.5355  R-03-low-overlap              441.56               0.9736                     0.00                      0.00           0.9736
lcdb1000034708 High Producing Exotic Grassland 390.8594  R-03-low-overlap            37681.40               0.9641                     0.00                      0.00           0.9641
lcdb1000035168 High Producing Exotic Grassland   2.1321  R-04-low-overlap                0.00               0.0000                   204.68                      0.96           0.9600
lcdb1000043194 High Producing Exotic Grassland  21.1674  R-03-low-overlap           

## Finding 2 — width is materially method-dependent

`2A/P` is an average-width proxy sensitive to compactness. The `-15 m` erosion
test asks a different question: does a 30 m-wide core survive anywhere in the
polygon? The disagreements are counted and their direction checked below.

In [4]:
disagreements = comparison[comparison.methods_disagree]
ap_fail_erosion_pass = (~disagreements.area_perimeter_pass) & disagreements.erosion_core_pass
ap_pass_erosion_fail = disagreements.area_perimeter_pass & (~disagreements.erosion_core_pass)

print(f'Width disagreements:     {len(disagreements):,} of {len(comparison):,} '
      f'({len(disagreements)/len(comparison):.2%})')
print(f'2A/P fail, erosion pass: {int(ap_fail_erosion_pass.sum()):,}')
print(f'2A/P pass, erosion fail: {int(ap_pass_erosion_fail.sum()):,}')
print()
print('2A/P width (m) among disagreements:')
print(disagreements.width_area_perimeter_m.describe()[['min', '50%', 'max']].to_string())

Width disagreements:     694 of 5,712 (12.15%)
2A/P fail, erosion pass: 694
2A/P pass, erosion fail: 0

2A/P width (m) among disagreements:
min    13.4080
50%    24.4905
max    29.9940


![Three real Gisborne disagreement cases](../outputs/gisborne/figures/width_disagreement_cases.png)

Every disagreement runs one way: a local 30 m core survives while the
compactness-sensitive value falls below 30 m. The examples show why a surviving
core cannot establish *average* width — long branching and dumbbell shapes keep
a core and still fail a centre-line average. All 694 are quarantined under R-02
rather than resolved by picking the more favourable proxy.

## The CRS failure mode is false rejection

The original project brief predicted that a degrees-versus-metres mistake would
let small polygons through. The real direction is the opposite.

In [5]:
nztm_pass = candidates.area >= 10_000
with warnings.catch_warnings():
    warnings.simplefilter('ignore', UserWarning)
    naive_wgs84_pass = candidates.to_crs(4326).area >= 10_000

print(f'NZTM2000 >=1 ha:       {int(nztm_pass.sum()):,}')
print(f'Naive WGS84 >=1 ha:    {int(naive_wgs84_pass.sum()):,}')
print(f'False rejections:      {int((nztm_pass & ~naive_wgs84_pass).sum()):,}')
print(f'False qualifications:  {int((~nztm_pass & naive_wgs84_pass).sum()):,}')

NZTM2000 >=1 ha:       4,223
Naive WGS84 >=1 ha:    0
False rejections:      4,223
False qualifications:  0


All 4,223 true area passes become false rejections, because a square degree
compared against 10,000 is a vanishingly small number. Nothing is falsely
qualified. The finding corrects the initial hypothesis instead of being fitted
to it, and every pipeline entry point now refuses input that is not EPSG:2193.

## Finding 3 — the imagery review is still outstanding

R-05 tests the mapped land-cover class, not what is on the ground. The fixed
sample of 30 candidates is pinned and rendered as imagery cards, but no label
file exists yet, so the findings carry a pending status and no rate.

In [6]:
pinned = pd.read_csv(OUT / 'review/review_sample_ids.csv')
summary = pd.read_csv(OUT / 'review/review_summary.csv')

print(f'pinned review sample: {len(pinned)} units')
print(f'first three:          {", ".join(pinned.unit_id.head(3))}')
print()
print(summary.to_string(index=False))
print()
print({k: v for k, v in findings.items() if k.startswith('visual_')})

pinned review sample: 30 units
first three:          lcdb1000012117, lcdb1000034948, lcdb1000035223

         metric                                                                                                                                        value
    sample_size                                                                                                                                           30
  review_status                                                                                                             pending_independent_human_review
labels_recorded                                                                                                                                            0
        imagery                                                                 Gisborne District Council Imagery Satellite Gisborne 2024 (credited to LINZ)
how_to_complete Copy review_labels_template.csv to review_labels.csv, label all 30 cards from review/cards/, then 

Any agreement or false-positive rate published from this sample must come from
a named person working through `outputs/gisborne/review/cards/`;
`scripts/ingest_review_labels.py` rejects a label file whose reviewer looks
automated.

## Reading the unit counts

Multipart LCDB units are exploded before screening, and each fragment is
screened independently under its own `-part-N` identifier. A count in this
notebook is therefore a count of screened fragments, not of source LCDB
polygons, and fragments of one polygon are not re-aggregated before R-01 or
R-02 are applied.